In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [5]:
data = pd.read_csv("symtom.csv")

data = data.dropna(subset=["text", "label"])

X = data["text"].astype(str).str.lower()
X = X.str.replace(r'[^a-zA-Z\s]', '', regex=True)
X = X.str.replace(r'\s+', ' ', regex=True).str.strip()
y = data["label"].astype(str)

In [6]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.20, random_state=42, stratify=y_encoded
)

In [7]:
vectorizer = TfidfVectorizer(max_features=3000, stop_words="english")
X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)
print('done')

done


In [12]:
models = {
    "Linear Support Vector (LinearSVC)": LinearSVC(random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=100, random_state=42),
    "Multinomial Naive Bayes": MultinomialNB(),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average="weighted", zero_division=0)
    rec = recall_score(y_test, y_pred, average="weighted", zero_division=0)
    f1 = f1_score(y_test, y_pred, average="weighted", zero_division=0)
    
    results.append({
        "Model": name,
        "Accuracy": f"{acc * 100:.2f}%",
        "Precision": f"{prec * 100:.2f}%",
        "Recall": f"{rec * 100:.2f}%",
        "F1-Score": f"{f1 * 100:.2f}%"
    })

comparison_df = pd.DataFrame(results)
print("=== ML ALGORITHMS PERFORMANCE COMPARISON ===")
print(comparison_df.to_string(index=False))

=== ML ALGORITHMS PERFORMANCE COMPARISON ===
                            Model Accuracy Precision Recall F1-Score
Linear Support Vector (LinearSVC)   95.42%    95.48% 95.42%   95.28%
              Logistic Regression   94.17%    94.56% 94.17%   94.05%
          Multinomial Naive Bayes   92.08%    92.91% 92.08%   91.86%
                    Random Forest   93.33%    94.31% 93.33%   93.34%


In [15]:
best_model = LinearSVC(random_state=42)
best_model.fit(X_train, y_train)
best_pred = best_model.predict(X_test)

print("==================================================")
print("BEST MODEL DETAILS (LinearSVC)")
print("==================================================")

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, best_pred))

# Get report as dictionary and convert to DataFrame
report_dict = classification_report(
    y_test, 
    best_pred, 
    target_names=label_encoder.classes_, 
    output_dict=True, 
    zero_division=0
)

report_df = pd.DataFrame(report_dict).transpose()

# Format metrics into percentages
report_df[["precision", "recall", "f1-score"]] = report_df[["precision", "recall", "f1-score"]].applymap(lambda x: f"{x * 100:.2f}%")
report_df["support"] = report_df["support"].astype(int)

print("\nClassification Report (%):\n")
print(report_df.to_string())

BEST MODEL DETAILS (LinearSVC)

Confusion Matrix:

[[10  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0 10  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0 10  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0 10  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  8  0  1  0  0  0  0  0  0  0  0  0  1  0  0  0  0  0  0  0]
 [ 0  0  0  0  0 10  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  2  0  8  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0 10  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0 10  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0 10  0  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0 10  0  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0 10  0  0  0  0  0  0  0  0  0  0  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0 10  0  0  0

/var/folders/cr/s1jcl2217530svy90n7rcyj40000gn/T/ipykernel_29145/738037303.py:24: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  report_df[["precision", "recall", "f1-score"]] = report_df[["precision", "recall", "f1-score"]].applymap(lambda x: f"{x * 100:.2f}%")
